In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loaders._load_vn30_binary import preprocess, VN30, TARGETS
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import TimeSeriesSplit

In [9]:
acc = []
for symbol in VN30:
    data = preprocess(symbol, lag=30)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]

    tscv = TimeSeriesSplit(n_splits=5)


In [10]:
X_train

,open_lag_0,high_lag_0,low_lag_0,close_lag_0,open_lag_1,open_lag_2,open_lag_3,open_lag_4,open_lag_5,open_lag_6,...,close_lag_21,close_lag_22,close_lag_23,close_lag_24,close_lag_25,close_lag_26,close_lag_27,close_lag_28,close_lag_29,close_lag_30
32,0.39,1.60,0.34,1.84,-0.05,0.73,-0.25,0.44,0.88,0.09,...,0.44,0.63,0.44,0.44,0.38,1.41,-0.29,0.39,-0.20,-1.06
33,2.24,1.90,1.99,0.98,0.39,-0.05,0.73,-0.25,0.44,0.88,...,-1.31,0.44,0.63,0.44,0.44,0.38,1.41,-0.29,0.39,-0.20
34,1.26,-0.15,-0.77,-1.07,2.24,0.39,-0.05,0.73,-0.25,0.44,...,-0.83,-1.31,0.44,0.63,0.44,0.44,0.38,1.41,-0.29,0.39
35,-1.75,-0.97,0.09,0.58,1.26,2.24,0.39,-0.05,0.73,-0.25,...,0.05,-0.83,-1.31,0.44,0.63,0.44,0.44,0.38,1.41,-0.29
36,1.07,0.39,1.12,0.00,-1.75,1.26,2.24,0.39,-0.05,0.73,...,-0.34,0.05,-0.83,-1.31,0.44,0.63,0.44,0.44,0.38,1.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,0.05,0.00,0.00,0.00,0.00,0.25,0.00,-0.20,-0.40,-0.40,...,-0.20,0.20,0.05,-0.25,-1.05,-0.25,0.10,0.50,-0.45,-0.35
1246,-0.10,0.40,0.15,0.45,0.05,0.00,0.25,0.00,-0.20,-0.40,...,0.00,-0.20,0.20,0.05,-0.25,-1.05,-0.25,0.10,0.50,-0.45
1247,0.50,0.10,0.30,-0.10,-0.10,0.05,0.00,0.25,0.00,-0.20,...,0.00,0.00,-0.20,0.20,0.05,-0.25,-1.05,-0.25,0.10,0.50
1248,-0.05,-0.05,0.05,-0.05,0.50,-0.10,0.05,0.00,0.25,0.00,...,0.20,0.00,0.00,-0.20,0.20,0.05,-0.25,-1.05,-0.25,0.10


In [11]:
Y_train

32      1
33      0
34      1
35      0
36      0
       ..
1245    1
1246    0
1247    0
1248    1
1249    0
Name: label, Length: 1218, dtype: int64

In [13]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
import numpy as np

def knn_pro_for_symbol(symbol, lag=30, use_volume=False):
    # Nếu muốn dùng volume: chỉnh sửa preprocess để không drop 'volume' và sinh lags/rolling tương tự OHLC.
    data = preprocess(symbol, lag=lag, use_rolling=True, use_calendar=True, feat_select=True)  # bật các feature mạnh
    (X_train, y_train) = data["train"]
    (X_test,  y_test)  = data["test"]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(svd_solver="full")),
        ("knn", KNeighborsClassifier())
    ])

    param_grid = {
        "pca__n_components": [None, 0.90, 0.95],
        "knn__n_neighbors": [3,5,7,11,15],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1,2],
    }

    cv = TimeSeriesSplit(n_splits=5)
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="balanced_accuracy", n_jobs=1, refit=True)
    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)
    ba = balanced_accuracy_score(y_test, y_pred)

    return grid.best_params_, grid.best_score_, ba

accs = []
for sym in VN30:
    best_params, cv_score, test_ba = knn_pro_for_symbol(sym, lag=30)
    print(f"[{sym}] test BA={test_ba:.3f} | cv={cv_score:.3f} | {best_params}")
    accs.append(test_ba)

print(f"\nMean BA across VN30: {np.mean(accs):.3f} ± {np.std(accs):.3f}")


[ACB] test BA=0.494 | cv=0.515 | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.9}
[BCM] test BA=0.532 | cv=0.526 | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.9}
[BID] test BA=0.480 | cv=0.517 | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.95}
[BVH] test BA=0.562 | cv=0.530 | {'knn__n_neighbors': 15, 'knn__p': 1, 'knn__weights': 'uniform', 'pca__n_components': None}
[CTG] test BA=0.490 | cv=0.511 | {'knn__n_neighbors': 15, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.9}
[FPT] test BA=0.532 | cv=0.530 | {'knn__n_neighbors': 11, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': None}
[GAS] test BA=0.517 | cv=0.499 | {'knn__n_neighbors': 11, 'knn__p': 1, 'knn__weights': 'uniform', 'pca__n_components': 0.95}
[GVR] test BA=0.512 | cv=0.507 | {'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'uniform', 'pca__n_components': None}
[HDB] t